# Vector Norms — companion notebook

Runnable companion for [`norms.md`](./norms.md).

We visualize unit balls for several $\ell_p$ norms, verify the norm ordering numerically, and demonstrate why $\ell_1$ regularization induces sparsity (the geometric explanation).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. Computing $\ell_p$ norms in NumPy

In [ ]:
x = np.array([3.0, -4.0])
print(f"L1   norm: {np.linalg.norm(x, ord=1):.4f}")
print(f"L2   norm: {np.linalg.norm(x, ord=2):.4f}")
print(f"Linf norm: {np.linalg.norm(x, ord=np.inf):.4f}")
print(f"L0 'norm': {np.count_nonzero(x)}    # not actually a norm")

## 2. Unit balls in $\mathbb{R}^2$

The unit ball $\{x : \|x\|_p \le 1\}$ visualizes the geometry of each norm.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
directions = np.stack([np.cos(theta), np.sin(theta)], axis=1)  # (400, 2)

fig, ax = plt.subplots(figsize=(6, 6))
for p, color in [(1, 'tab:blue'), (2, 'tab:orange'), (np.inf, 'tab:green'), (0.5, 'tab:red')]:
    # For each direction d, find scale s such that ||s*d||_p = 1.
    if p == 0.5:  # not a true norm; compute the "quasi-norm" manually
        norms = (np.abs(directions[:, 0])**0.5 + np.abs(directions[:, 1])**0.5)**2
    else:
        norms = np.linalg.norm(directions, ord=p, axis=1)
    points = directions / norms[:, None]
    label = f"p = {p}" if np.isfinite(p) else "p = inf"
    ax.plot(points[:, 0], points[:, 1], color=color, label=label)

ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_title(r"Unit balls $\{x : \|x\|_p = 1\}$ in $\mathbb{R}^2$")
ax.legend()
plt.show()

Notice:
* $p=1$ is a diamond — corners on the axes.
* $p=2$ is the familiar circle.
* $p=\infty$ is a square.
* $p=0.5$ is non-convex (and not a true norm because the triangle inequality fails).

## 3. Verifying the norm ordering

$\|x\|_\infty \le \|x\|_2 \le \|x\|_1 \le \sqrt{n} \|x\|_2 \le n \|x\|_\infty$ for any $x \in \mathbb{R}^n$.

In [ ]:
n = 50
for _ in range(5):
    x = rng.standard_normal(n)
    l_inf = np.linalg.norm(x, ord=np.inf)
    l_2 = np.linalg.norm(x, ord=2)
    l_1 = np.linalg.norm(x, ord=1)
    assert l_inf <= l_2 + 1e-12 <= l_1 + 1e-12 <= np.sqrt(n) * l_2 + 1e-12 <= n * l_inf + 1e-12
    print(f"Linf={l_inf:.3f}  L2={l_2:.3f}  L1={l_1:.3f}  sqrt(n)L2={np.sqrt(n)*l_2:.3f}  nLinf={n*l_inf:.3f}")
print("\nAll inequalities hold.")

## 4. Why $\ell_1$ induces sparsity — the geometric picture

Constrained least squares: $\min_\beta \|y - X\beta\|_2^2$ subject to $\|\beta\|_p \le t$.

The level sets of the loss are ellipses; the constraint set is the unit ball of the chosen norm (scaled by $t$). The solution lies where the lowest-value loss ellipse first touches the constraint. The **corners** of the $\ell_1$ ball sit on the axes, so the contact point is very often *on* an axis — meaning some coordinates of $\beta$ are exactly zero. The smooth $\ell_2$ ball has no corners, so the contact point almost never lies on an axis.

In [ ]:
# Quadratic loss with optimum off-axis
beta_star = np.array([2.0, 0.7])
A = np.array([[1.0, 0.3], [0.3, 2.0]])  # PSD

def loss(b):
    d = b - beta_star
    return d @ A @ d

grid = np.linspace(-1.5, 3.5, 400)
B1, B2 = np.meshgrid(grid, grid)
Z = np.vectorize(lambda a, b: loss(np.array([a, b])))(B1, B2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
theta = np.linspace(0, 2 * np.pi, 400)
d = np.stack([np.cos(theta), np.sin(theta)], axis=1)
t = 1.0
for ax, p, name in zip(axes, [1, 2], ['L1 ball (diamond)', 'L2 ball (circle)']):
    ax.contour(B1, B2, Z, levels=15, cmap='Greys', alpha=0.6)
    norms = np.linalg.norm(d, ord=p, axis=1)
    ball = t * d / norms[:, None]
    ax.plot(ball[:, 0], ball[:, 1], 'tab:red', lw=2)
    ax.scatter(*beta_star, color='tab:blue', label='unconstrained optimum')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal')
    ax.set_title(name)
    ax.legend()
plt.show()

Look at where the loss contours first kiss the red constraint set. With the $\ell_1$ diamond the contact point sits on a corner (one coordinate = 0); with the $\ell_2$ circle the contact is interior to a quadrant (both coordinates nonzero). That is the entire geometric reason lasso produces sparse solutions and ridge does not.